# STEP 1A.2 — Targeted Adjacent-Literature Follow-up

This notebook follows the broad Step 1A open-discovery pass.

The broad pass successfully surfaced several potentially important neighbouring literatures, but OpenAlex retrieval was very noisy because many queries returned thousands to millions of records and only the top-ranked 100 were sampled.

This follow-up therefore does **not** repeat broad discovery.

It performs targeted searches of the specific adjacent traditions that emerged as potentially relevant:

1. intelligent tutoring / adaptive learning / learner modelling / knowledge tracing;
2. JITAIs and state-contingent adaptive interventions;
3. performative prediction and dynamical feedback;
4. selective exposure / information avoidance / confirmatory search;
5. agency, active/passive acquisition, hybrid human-AI regulation;
6. algorithm awareness / perceived personalisation;
7. adaptive assessment / adaptive testing;
8. open learner models / learning analytics / self-regulated learning.

The purpose is to decide later whether these are:
- load-bearing literature domains;
- useful subdomains of existing Step-1 domains;
- or merely adjacent background.

The notebook only harvests and organises. It does not make that judgement.

## 1. Install and mount Drive

In [ ]:
!pip -q install pandas numpy requests openpyxl biopython rapidfuzz

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import requests, re, time, os, json
from rapidfuzz import fuzz
from Bio import Entrez

EMAIL = "your_email@example.com"  # CHANGE THIS
Entrez.email = EMAIL

ROOT = Path('/content/drive/MyDrive/Step1_Evidence_Corpus')
ROOT.mkdir(parents=True, exist_ok=True)

MAX_OPENALEX = 75
MAX_PUBMED = 75
FORCE_RERUN = False

print(ROOT)

## 2. Targeted search plan

OpenAlex and PubMed use separate query syntax where useful. These queries are intentionally narrower than Step 1A.

In [ ]:
TARGETED_PLAN = [
    # Adaptive education / ITS
    {
        "id":"ODF001","cluster":"adaptive education / ITS",
        "oa":'"intelligent tutoring system" adaptive learner model problem selection',
        "pm":'("intelligent tutoring system"[Title/Abstract] OR "adaptive learning"[Title/Abstract]) AND ("learner model"[Title/Abstract] OR "problem selection"[Title/Abstract] OR "adaptive content"[Title/Abstract])',
        "reason":"State-driven future content selection."
    },
    {
        "id":"ODF002","cluster":"adaptive education / ITS",
        "oa":'"knowledge tracing" adaptive problem selection tutoring',
        "pm":'"knowledge tracing"[Title/Abstract] AND (adaptive[Title/Abstract] OR "problem selection"[Title/Abstract] OR tutoring[Title/Abstract])',
        "reason":"Learner-state inference linked to content/item selection."
    },
    {
        "id":"ODF003","cluster":"adaptive education / ITS",
        "oa":'"adaptive scaffolding" learner behavior intelligent tutoring experiment',
        "pm":'"adaptive scaffolding"[Title/Abstract] AND (learning[Title/Abstract] OR tutoring[Title/Abstract])',
        "reason":"Adaptive support contingent on evolving learner state."
    },
    {
        "id":"ODF004","cluster":"adaptive education / ITS",
        "oa":'"personalized adaptive learning" experimental learning outcomes',
        "pm":'"personalized adaptive learning"[Title/Abstract] AND (experiment*[Title/Abstract] OR random*[Title/Abstract] OR learning[Title/Abstract])',
        "reason":"Human outcome evidence rather than platform descriptions."
    },

    # JITAI/adaptive intervention
    {
        "id":"ODF005","cluster":"JITAI / adaptive intervention",
        "oa":'"just-in-time adaptive intervention" decision points tailoring variables proximal outcomes',
        "pm":'"just-in-time adaptive intervention"[Title/Abstract] AND ("decision point"[Title/Abstract] OR "tailoring variable"[Title/Abstract] OR "proximal outcome"[Title/Abstract])',
        "reason":"Formal state-contingent intervention architecture."
    },
    {
        "id":"ODF006","cluster":"JITAI / adaptive intervention",
        "oa":'"micro-randomized trial" tailoring variable proximal causal effect',
        "pm":'"micro-randomized trial"[Title/Abstract] AND ("proximal effect"[Title/Abstract] OR "tailoring variable"[Title/Abstract])',
        "reason":"Identification of time-varying adaptive components."
    },
    {
        "id":"ODF007","cluster":"JITAI / adaptive intervention",
        "oa":'"adaptive intervention" changing state personalized support repeated decision',
        "pm":'"adaptive intervention"[Title/Abstract] AND ("changing state"[Title/Abstract] OR "time-varying"[Title/Abstract]) AND personalized[Title/Abstract]',
        "reason":"Broader state-contingent adaptive-intervention vocabulary."
    },

    # Performative prediction/dynamical feedback
    {
        "id":"ODF008","cluster":"performative prediction / dynamic feedback",
        "oa":'"performative prediction" feedback decision distribution',
        "pm":'"performative prediction"[Title/Abstract]',
        "reason":"Prediction/action changes future data distribution."
    },
    {
        "id":"ODF009","cluster":"performative prediction / dynamic feedback",
        "oa":'"performative stability" prediction distribution shift',
        "pm":'"performative stability"[Title/Abstract]',
        "reason":"Formal stability of endogenous prediction environments."
    },
    {
        "id":"ODF010","cluster":"performative prediction / dynamic feedback",
        "oa":'"decision-dependent distribution" machine learning feedback',
        "pm":'"decision-dependent distribution"[Title/Abstract] AND feedback[Title/Abstract]',
        "reason":"Alternative terminology for endogenous distribution shift."
    },
    {
        "id":"ODF011","cluster":"performative prediction / dynamic feedback",
        "oa":'"feedback loops" automated decision-making dynamical systems bias',
        "pm":'("feedback loop"[Title/Abstract] OR "feedback loops"[Title/Abstract]) AND "automated decision"[Title/Abstract]',
        "reason":"Taxonomies of dynamic machine-decision feedback."
    },

    # Selective exposure / information avoidance
    {
        "id":"ODF012","cluster":"selective exposure / information avoidance",
        "oa":'"information avoidance" belief updating experiment',
        "pm":'"information avoidance"[Title/Abstract] AND (belief*[Title/Abstract] OR learning[Title/Abstract])',
        "reason":"Information utility may include avoidance, not only uncertainty reduction."
    },
    {
        "id":"ODF013","cluster":"selective exposure / information avoidance",
        "oa":'"selective exposure" confirmation bias information search experiment',
        "pm":'"selective exposure"[Title/Abstract] AND ("confirmation bias"[Title/Abstract] OR "information search"[Title/Abstract])',
        "reason":"Self-confirming evidence acquisition."
    },
    {
        "id":"ODF014","cluster":"selective exposure / information avoidance",
        "oa":'"positive test strategy" hypothesis testing information search',
        "pm":'"positive test strategy"[Title/Abstract] AND (hypothesis[Title/Abstract] OR information[Title/Abstract])',
        "reason":"Cognitive hypothesis-testing literature."
    },
    {
        "id":"ODF015","cluster":"selective exposure / information avoidance",
        "oa":'"confirmatory information search" learning beliefs experiment',
        "pm":'"confirmatory information search"[Title/Abstract]',
        "reason":"Alternative confirmatory-search terminology."
    },

    # Agency / hybrid human-AI
    {
        "id":"ODF016","cluster":"agency / active-passive acquisition",
        "oa":'"active learning" "passive learning" agency memory experiment self selection',
        "pm":'("active learning"[Title/Abstract] AND "passive learning"[Title/Abstract]) AND (agency[Title/Abstract] OR memory[Title/Abstract] OR selection[Title/Abstract])',
        "reason":"Psychological consequence of self-directed acquisition."
    },
    {
        "id":"ODF017","cluster":"agency / active-passive acquisition",
        "oa":'"self-generation effect" choice learning memory experiment',
        "pm":'("self-generation"[Title/Abstract] OR "generation effect"[Title/Abstract]) AND (choice[Title/Abstract] OR learning[Title/Abstract] OR memory[Title/Abstract])',
        "reason":"Self-generation as alternative to contingency."
    },
    {
        "id":"ODF018","cluster":"hybrid human-AI / interactive ML",
        "oa":'"interactive machine learning" human behavior feedback learning system',
        "pm":'"interactive machine learning"[Title/Abstract] AND human[Title/Abstract]',
        "reason":"Human and algorithm jointly changing system state."
    },
    {
        "id":"ODF019","cluster":"hybrid human-AI / interactive ML",
        "oa":'"hybrid human-AI regulation" adaptive learning self-regulated learning',
        "pm":'("hybrid human-AI"[Title/Abstract] OR "hybrid intelligence"[Title/Abstract]) AND "self-regulated learning"[Title/Abstract]',
        "reason":"Shared/shifted regulation between adaptive system and learner."
    },

    # Algorithm awareness / perceived personalization
    {
        "id":"ODF020","cluster":"algorithm awareness / perceived personalisation",
        "oa":'"algorithm awareness" personalized recommendation behavior',
        "pm":'"algorithm awareness"[Title/Abstract] AND (recommend*[Title/Abstract] OR personali*[Title/Abstract])',
        "reason":"Awareness as rival pathway."
    },
    {
        "id":"ODF021","cluster":"algorithm awareness / perceived personalisation",
        "oa":'"perceived personalization" recommendation behavior experiment',
        "pm":'"perceived personalization"[Title/Abstract] AND (recommend*[Title/Abstract] OR behavior[Title/Abstract])',
        "reason":"Perceived rather than actual personalization."
    },
    {
        "id":"ODF022","cluster":"algorithm awareness / perceived personalisation",
        "oa":'"personalization cues" algorithm user behavior',
        "pm":'"personalization cue"[Title/Abstract] OR "personalization cues"[Title/Abstract]',
        "reason":"Experimental cues that can change behavior absent policy differences."
    },
    {
        "id":"ODF023","cluster":"algorithm awareness / perceived personalisation",
        "oa":'"perceived algorithmic responsiveness" personalization',
        "pm":'"algorithmic responsiveness"[Title/Abstract]',
        "reason":"Perceived contingency/responsiveness terminology."
    },

    # Adaptive assessment
    {
        "id":"ODF024","cluster":"adaptive assessment",
        "oa":'"computerized adaptive testing" learning testing effect item selection',
        "pm":'"computerized adaptive testing"[Title/Abstract] AND (learning[Title/Abstract] OR "testing effect"[Title/Abstract])',
        "reason":"State-contingent assessment and possible reactivity."
    },
    {
        "id":"ODF025","cluster":"adaptive assessment",
        "oa":'"adaptive assessment" item selection learning reactivity',
        "pm":'"adaptive assessment"[Title/Abstract] AND ("item selection"[Title/Abstract] OR learning[Title/Abstract])',
        "reason":"Adaptive item selection as measurement environment."
    },

    # Open learner models / SRL
    {
        "id":"ODF026","cluster":"open learner models / self regulation",
        "oa":'"open learner model" adaptive tutoring self-regulated learning',
        "pm":'"open learner model"[Title/Abstract] AND ("self-regulated learning"[Title/Abstract] OR adaptive[Title/Abstract])',
        "reason":"Learner access to system state/model may change regulation."
    },
    {
        "id":"ODF027","cluster":"open learner models / self regulation",
        "oa":'"MetaTutor" adaptive scaffolding self-regulated learning',
        "pm":'MetaTutor[Title/Abstract] AND "self-regulated learning"[Title/Abstract]',
        "reason":"Long-running adaptive SRL research programme."
    },
    {
        "id":"ODF028","cluster":"open learner models / self regulation",
        "oa":'"learning analytics" learner model adaptive intervention self regulation',
        "pm":'"learning analytics"[Title/Abstract] AND "learner model"[Title/Abstract] AND adaptive[Title/Abstract]',
        "reason":"Behaviour-derived learner state feeding adaptive support."
    }
]

plan_df = pd.DataFrame(TARGETED_PLAN)
display(plan_df)
print("Queries:",len(plan_df),"x 2 databases")

## 3. Load persistent corpus tables

In [ ]:
QUERY_PLAN_COLUMNS = [
    "query_plan_id","domain","search_purpose","planned_query",
    "preferred_sources","priority","rationale","status","notes"
]
SEARCH_COLUMNS = [
    "search_id","query_plan_id","domain","database","date_searched",
    "query","filters","search_purpose","direct_or_adjacent",
    "total_results_reported","results_retrieved","results_screened",
    "backward_checked","forward_checked","notes"
]
RETRIEVAL_COLUMNS = [
    "retrieval_id","search_id","candidate_id","retrieval_rank",
    "retrieval_method","database_record_id","notes"
]
CANDIDATE_COLUMNS = [
    "candidate_id","title","authors","year","doi","url",
    "screening_decision","exclusion_reason","duplicate_of",
    "linked_publication_id","notes"
]
DISCOVERY_COLUMNS = [
    "discovery_record_id","candidate_id","database","database_record_id",
    "abstract","subject_terms","source_topics","publication_type",
    "cited_by_count","retrieved_date","notes"
]

SCHEMAS = {
    "query_plan":QUERY_PLAN_COLUMNS,
    "searches":SEARCH_COLUMNS,
    "retrievals":RETRIEVAL_COLUMNS,
    "candidates":CANDIDATE_COLUMNS,
    "discovery_records":DISCOVERY_COLUMNS
}

def p(name): return ROOT/f"{name}.csv"

def load(name,cols):
    if p(name).exists():
        df=pd.read_csv(p(name),dtype=str).fillna("")
        for c in cols:
            if c not in df.columns: df[c]=""
        return df[cols]
    df=pd.DataFrame(columns=cols); df.to_csv(p(name),index=False); return df

tables={n:load(n,c) for n,c in SCHEMAS.items()}

existing=set(tables["query_plan"]["query_plan_id"].astype(str))
add=[]
for q in TARGETED_PLAN:
    if q["id"] not in existing:
        add.append({
            "query_plan_id":q["id"],
            "domain":q["cluster"],
            "search_purpose":"TARGETED_OPEN_DISCOVERY",
            "planned_query":q["oa"],
            "preferred_sources":"OpenAlex;PubMed",
            "priority":"HIGH",
            "rationale":q["reason"],
            "status":"PLANNED",
            "notes":"Step 1A.2 targeted adjacent-literature follow-up"
        })
if add:
    tables["query_plan"]=pd.concat([tables["query_plan"],pd.DataFrame(add)],ignore_index=True)
    tables["query_plan"].to_csv(p("query_plan"),index=False)

print({k:len(v) for k,v in tables.items()})

## 4. IDs and normalization

In [ ]:
PREFIX={"searches":"S","retrievals":"RET","candidates":"C","discovery_records":"DR"}
IDCOL={"searches":"search_id","retrievals":"retrieval_id","candidates":"candidate_id","discovery_records":"discovery_record_id"}

def save(name): tables[name].to_csv(p(name),index=False)

def next_id(name):
    nums=[]
    for v in tables[name][IDCOL[name]].astype(str):
        m=re.search(r"(\d+)$",v)
        if m: nums.append(int(m.group(1)))
    return f"{PREFIX[name]}{max(nums,default=0)+1:05d}"

def normdoi(x):
    x=str(x or "").strip().lower()
    x=re.sub(r"^https?://(dx\.)?doi\.org/","",x)
    x=re.sub(r"^doi:\s*","",x)
    return x.strip()

def normtitle(x):
    return re.sub(r"[^a-z0-9]+"," ",str(x or "").lower()).strip()

def st(x): return "" if x is None else str(x)

## 5. Retrieval functions

In [ ]:
def pubmed_records(query,retmax=75):
    h=Entrez.esearch(db="pubmed",term=query,retmax=retmax,sort="relevance")
    sr=Entrez.read(h)
    ids=list(sr.get("IdList",[])); total=int(sr.get("Count",0))
    if not ids: return total,pd.DataFrame()
    time.sleep(.35)
    h=Entrez.efetch(db="pubmed",id=",".join(ids),rettype="xml")
    data=Entrez.read(h)
    rows=[]
    for pa in data.get("PubmedArticle",[]):
        try:
            med=pa["MedlineCitation"]; art=med["Article"]
            pmid=st(med.get("PMID","")); title=st(art.get("ArticleTitle",""))
            abs_parts=[]
            ao=art.get("Abstract",{})
            for part in ao.get("AbstractText",[]) if ao else []:
                label=getattr(part,"attributes",{}).get("Label","")
                abs_parts.append((label+": " if label else "")+st(part))
            authors=[]
            for a in art.get("AuthorList",[]) or []:
                coll=st(a.get("CollectiveName",""))
                if coll: authors.append(coll)
                else:
                    nm=" ".join(x for x in [st(a.get("LastName","")),st(a.get("Initials",""))] if x)
                    if nm: authors.append(nm)
            doi=""
            for aid in pa.get("PubmedData",{}).get("ArticleIdList",[]):
                if getattr(aid,"attributes",{}).get("IdType")=="doi":
                    doi=normdoi(aid); break
            ji=art.get("Journal",{}).get("JournalIssue",{})
            pdte=ji.get("PubDate",{}) if ji else {}
            year=st(pdte.get("Year",""))
            if not year:
                m=re.search(r"(19|20)\d{2}",st(pdte.get("MedlineDate","")))
                year=m.group(0) if m else ""
            mesh=[]
            for mh in med.get("MeshHeadingList",[]) or []:
                d=st(mh.get("DescriptorName",""))
                if d: mesh.append(d)
            ptypes=[st(x) for x in art.get("PublicationTypeList",[]) or []]
            rows.append({
                "database":"PubMed","database_record_id":pmid,
                "title":title,"authors":"; ".join(authors),"year":year,
                "doi":doi,"url":f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
                "abstract":" ".join(abs_parts),
                "subject_terms":"; ".join(sorted(set(mesh))),
                "source_topics":"","publication_type":"; ".join(ptypes),
                "cited_by_count":""
            })
        except Exception:
            pass
    return total,pd.DataFrame(rows)

def oa_abstract(inv):
    if not inv: return ""
    mx=max((max(v) for v in inv.values() if v),default=-1)
    if mx<0:return ""
    a=[""]*(mx+1)
    for word,pos in inv.items():
        for j in pos:
            if 0<=j<len(a):a[j]=word
    return " ".join(x for x in a if x)

def openalex_records(query,max_records=75):
    rows=[]; cursor="*"; total=None
    while len(rows)<max_records:
        r=requests.get("https://api.openalex.org/works",params={
            "search":query,"per-page":min(100,max_records-len(rows)),
            "cursor":cursor,"mailto":EMAIL
        },timeout=45)
        r.raise_for_status(); data=r.json()
        if total is None: total=data.get("meta",{}).get("count",0)
        batch=data.get("results",[])
        if not batch:break
        for w in batch:
            auth=[(a.get("author") or {}).get("display_name","") for a in w.get("authorships",[]) or []]
            topics=[x.get("display_name","") for x in (w.get("topics",[]) or [])[:10] if x.get("display_name")]
            kws=[x.get("display_name","") for x in (w.get("keywords",[]) or [])[:20] if x.get("display_name")]
            rows.append({
                "database":"OpenAlex","database_record_id":st(w.get("id","")),
                "title":st(w.get("display_name","")),"authors":"; ".join(x for x in auth if x),
                "year":st(w.get("publication_year","")),"doi":normdoi(w.get("doi","")),
                "url":st((w.get("primary_location") or {}).get("landing_page_url","")),
                "abstract":oa_abstract(w.get("abstract_inverted_index")),
                "subject_terms":"; ".join(kws),"source_topics":"; ".join(topics),
                "publication_type":st(w.get("type","")),
                "cited_by_count":st(w.get("cited_by_count",""))
            })
            if len(rows)>=max_records:break
        cursor=data.get("meta",{}).get("next_cursor")
        if not cursor:break
        time.sleep(.15)
    return int(total or 0),pd.DataFrame(rows)

## 6. Persistent storage

In [ ]:
def searched(qid,db):
    if FORCE_RERUN:return False
    s=tables["searches"]
    return len(s) and ((s["query_plan_id"].astype(str)==qid)&(s["database"].astype(str)==db)).any()

def add_search(q,db,query,total,n,err=""):
    sid=next_id("searches")
    row={
        "search_id":sid,"query_plan_id":q["id"],"domain":q["cluster"],
        "database":db,"date_searched":pd.Timestamp.now().date().isoformat(),
        "query":query,"filters":"","search_purpose":"TARGETED_OPEN_DISCOVERY",
        "direct_or_adjacent":"TARGETED_ADJACENT","total_results_reported":st(total),
        "results_retrieved":st(n),"results_screened":"0",
        "backward_checked":"","forward_checked":"","notes":err
    }
    tables["searches"]=pd.concat([tables["searches"],pd.DataFrame([row])],ignore_index=True);save("searches")
    return sid

def existing_candidate(title,doi):
    c=tables["candidates"]; doi=normdoi(doi)
    if len(c) and doi:
        x=c[c["doi"].astype(str).map(normdoi)==doi]
        if len(x):return x.iloc[0]["candidate_id"]
    nt=normtitle(title)
    if len(c) and nt:
        x=c[c["title"].astype(str).map(normtitle)==nt]
        if len(x):return x.iloc[0]["candidate_id"]
    return None

def candidate(row):
    old=existing_candidate(row.get("title",""),row.get("doi",""))
    if old:return old
    cid=next_id("candidates")
    new={
        "candidate_id":cid,"title":row.get("title",""),"authors":row.get("authors",""),
        "year":row.get("year",""),"doi":normdoi(row.get("doi","")),"url":row.get("url",""),
        "screening_decision":"NOT_SCREENED","exclusion_reason":"","duplicate_of":"",
        "linked_publication_id":"","notes":"Step 1A.2 targeted discovery"
    }
    tables["candidates"]=pd.concat([tables["candidates"],pd.DataFrame([new])],ignore_index=True);save("candidates")
    return cid

def retrieval(sid,cid,row,rank):
    rid=next_id("retrievals")
    new={
        "retrieval_id":rid,"search_id":sid,"candidate_id":cid,
        "retrieval_rank":st(rank),"retrieval_method":"targeted adjacent-literature search",
        "database_record_id":row.get("database_record_id",""),"notes":""
    }
    tables["retrievals"]=pd.concat([tables["retrievals"],pd.DataFrame([new])],ignore_index=True);save("retrievals")

def discovery(cid,row):
    d=tables["discovery_records"]
    same=(d["candidate_id"].astype(str)==cid)&(d["database"].astype(str)==row.get("database",""))&(d["database_record_id"].astype(str)==row.get("database_record_id","")) if len(d) else pd.Series(dtype=bool)
    if len(d) and same.any():return
    did=next_id("discovery_records")
    new={
        "discovery_record_id":did,"candidate_id":cid,"database":row.get("database",""),
        "database_record_id":row.get("database_record_id",""),
        "abstract":row.get("abstract",""),"subject_terms":row.get("subject_terms",""),
        "source_topics":row.get("source_topics",""),"publication_type":row.get("publication_type",""),
        "cited_by_count":row.get("cited_by_count",""),
        "retrieved_date":pd.Timestamp.now().date().isoformat(),
        "notes":"Discovery metadata only"
    }
    tables["discovery_records"]=pd.concat([tables["discovery_records"],pd.DataFrame([new])],ignore_index=True);save("discovery_records")

## 7. Run all targeted searches

In [ ]:
run_rows=[]
for q in TARGETED_PLAN:
    for db in ["OpenAlex","PubMed"]:
        if searched(q["id"],db):
            print("SKIP",q["id"],db)
            continue
        query=q["oa"] if db=="OpenAlex" else q["pm"]
        print("\nRUN",q["id"],db,query)
        try:
            total,hits=(openalex_records(query,MAX_OPENALEX) if db=="OpenAlex" else pubmed_records(query,MAX_PUBMED))
            sid=add_search(q,db,query,total,len(hits))
            for rank,(_,row) in enumerate(hits.iterrows(),1):
                cid=candidate(row); retrieval(sid,cid,row,rank); discovery(cid,row)
            run_rows.append({"query":q["id"],"cluster":q["cluster"],"database":db,"total":total,"retrieved":len(hits),"status":"OK"})
            print(" total",total,"retrieved",len(hits))
        except Exception as e:
            add_search(q,db,query,"",0,"ERROR: "+str(e))
            run_rows.append({"query":q["id"],"cluster":q["cluster"],"database":db,"total":"","retrieved":0,"status":"ERROR: "+str(e)})
            print(" ERROR",e)
        time.sleep(.35)

run_summary=pd.DataFrame(run_rows)
display(run_summary)

## 8. Build targeted cluster report

In [ ]:
def targeted_report_rows():
    s=tables["searches"]
    t=s[s["search_purpose"].astype(str)=="TARGETED_OPEN_DISCOVERY"].copy()
    r=tables["retrievals"]
    c=tables["candidates"]
    d=tables["discovery_records"]
    if not len(t):return pd.DataFrame()
    x=r.merge(t[["search_id","query_plan_id","domain","database","query"]],on="search_id",how="inner")
    x=x.merge(c,on="candidate_id",how="left",suffixes=("","_cand"))
    # aggregate retrieval breadth
    rows=[]
    for cid,g in x.groupby("candidate_id"):
        cr=g.iloc[0]
        dr=d[d["candidate_id"].astype(str)==str(cid)]
        abstracts=" ".join(dict.fromkeys([z for z in dr["abstract"].astype(str) if z.strip()]))
        subjects=[]
        topics=[]
        for z in dr["subject_terms"].astype(str):
            subjects.extend([v.strip() for v in z.split(";") if v.strip()])
        for z in dr["source_topics"].astype(str):
            topics.extend([v.strip() for v in z.split(";") if v.strip()])
        rows.append({
            "candidate_id":cid,
            "title":cr["title"],
            "year":cr["year"],
            "doi":cr["doi"],
            "clusters":"; ".join(sorted(set(g["domain"].astype(str)))),
            "query_count":g["query_plan_id"].nunique(),
            "databases":"; ".join(sorted(set(g["database"].astype(str)))),
            "abstract":abstracts,
            "subject_terms":"; ".join(sorted(set(subjects))),
            "source_topics":"; ".join(sorted(set(topics))),
            "manual_relevance":"",
            "manual_new_domain_or_subdomain":"",
            "manual_followup_action":"",
            "notes":""
        })
    return pd.DataFrame(rows).sort_values(["query_count","year"],ascending=[False,False])

target_candidates=targeted_report_rows()
print("Unique targeted candidates:",len(target_candidates))
display(target_candidates.head(50))

## 9. Cluster diagnostics

In [ ]:
def cluster_diagnostics():
    s=tables["searches"]
    s=s[s["search_purpose"].astype(str)=="TARGETED_OPEN_DISCOVERY"]
    r=tables["retrievals"]
    rows=[]
    for cluster in sorted(set(q["cluster"] for q in TARGETED_PLAN)):
        ss=s[s["domain"].astype(str)==cluster]
        ids=set(ss["search_id"].astype(str))
        rr=r[r["search_id"].astype(str).isin(ids)]
        rows.append({
            "cluster":cluster,
            "search_executions":len(ss),
            "reported_hits_sum":sum(int(x) for x in ss["total_results_reported"] if str(x).isdigit()),
            "retrievals":len(rr),
            "unique_candidates":rr["candidate_id"].nunique(),
            "pubmed_retrievals":len(rr.merge(ss[ss["database"]=="PubMed"][["search_id"]],on="search_id",how="inner")),
            "openalex_retrievals":len(rr.merge(ss[ss["database"]=="OpenAlex"][["search_id"]],on="search_id",how="inner"))
        })
    return pd.DataFrame(rows)

cluster_diag=cluster_diagnostics()
display(cluster_diag)

## 10. Export

In [ ]:
out=ROOT/"Step1A2_Targeted_Discovery_Report.xlsx"

s=tables["searches"]
ts=s[s["search_purpose"].astype(str)=="TARGETED_OPEN_DISCOVERY"].copy()
ids=set(ts["search_id"].astype(str))
tr=tables["retrievals"][tables["retrievals"]["search_id"].astype(str).isin(ids)].copy()
cids=set(tr["candidate_id"].astype(str))
tc=tables["candidates"][tables["candidates"]["candidate_id"].astype(str).isin(cids)].copy()
td=tables["discovery_records"][tables["discovery_records"]["candidate_id"].astype(str).isin(cids)].copy()

with pd.ExcelWriter(out,engine="openpyxl") as w:
    plan_df.to_excel(w,sheet_name="Targeted_Plan",index=False)
    ts.to_excel(w,sheet_name="Searches",index=False)
    tr.to_excel(w,sheet_name="Retrievals",index=False)
    tc.to_excel(w,sheet_name="Candidates",index=False)
    td.to_excel(w,sheet_name="Discovery_Metadata",index=False)
    target_candidates.to_excel(w,sheet_name="Targeted_Candidate_Report",index=False)
    cluster_diag.to_excel(w,sheet_name="Cluster_Diagnostics",index=False)
    run_summary.to_excel(w,sheet_name="Run_Summary",index=False)

print("EXPORT:",out)
print("Upload Step1A2_Targeted_Discovery_Report.xlsx back to the chat.")